<a href="https://colab.research.google.com/github/ecuirty-kr/1_DataAnalysis/blob/main/ch5_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[구글 코랩(Colab)에서 실행하기](https://colab.research.google.com/github/lovedlim/bae-v3/blob/main/part3/ch5/ch5_logistic_regression.ipynb)

# 로지스틱 회귀분석

In [10]:
## 30분 내로 로지 회귀 마치기 - 오늘 작업 3 유형 / 기출 2, 3 위주로 진행하고 / 나머지 1유형 + 안 된 부분(암기) 하면 시험 보겟네
## 로지스틱 회귀 모델 : 종속 변수가 두 범주로 구성된 경우 + 독립 변수 여러 개
## statsmodels 라이브러리 logit(로지스틱이니까 로짓) 사용

### 문제 조건 : 특정 질병의 유무를 나타내는 환자 데이터 셋. 로지스틱 회귀 모델을 사용하여 age, bmi, smoker, lifestyle을 독립 변수로
### 사용하여 질병의 발생 여부를 예측하는 모델을 적합하시오. bmi 변수의 계수 값?
### 종속 변수가 이진 형태(0, 1) -> 특정 질병의 유무(0:없음 / 1:있음) 등 logit() 함수에 적합
## 데이터 ) 독립변수 : age, bmi, smoker, activity_level, lifestyle  /  종속변수 : 특정 질병의 유무

# 1. 라이브러리
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part3/ch5/health_survey2.csv") # 링크는 모다? 복붙
print(df.head(5)) # 데이터 확인

# 2. 로지스틱 회귀 모델 (logit 함수)
from statsmodels.formula.api import logit ## ols와 동일한 라이브러리

model = logit('disease~bmi+smoker+age+lifestyle', data=df).fit() ## 있는 컬럼명 다 적지말고, 문제를 제대로 읽어
print(model.summary()) ## 통계적 요약 확인
## 테이블에서 확인할 것: coef (회귀 계수) - bmi 회귀계수(coef) 값 = 0.0563 (문제에서 bmi 변수의 계수 값 질의)

   age   bmi  smoker  activity_level  disease lifestyle
0   62  35.2       0               0        1         S
1   65  18.6       0               2        1         A
2   71  33.2       0               1        1         M
3   18  37.1       1               2        0         A
4   21  17.6       0               0        0         S
Optimization terminated successfully.
         Current function value: 0.641392
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                disease   No. Observations:                 1000
Model:                          Logit   Df Residuals:                      994
Method:                           MLE   Df Model:                            5
Date:                Tue, 16 Jun 2026   Pseudo R-squ.:                 0.05341
Time:                        17:45:54   Log-Likelihood:                -641.39
converged:                       True   LL-Null:                       -677.58
Covarianc

In [15]:
#### 오즈 / 오즈비
### 오즈 : 어떠한 사건이 발생할 확률과 그 사건이 발생하지 않을 확률의 비율 (범위: 0~무한)
### 오즈비 : 특정 두 그룹의 오즈를 비교한 값 (독립 변수 변화에 따른 종속 변수 발생의 오즈 변화를 비교할 예정)
### 오즈비는 1을 기준값으로 삼음. 1보다 크면 그만큼의 배수로 계산 / 1 이면 영향 없음 / 1보다 작으면 감소

## 문제 조건 : 위 1번 문제에서 적합한 로지스틱 회귀 모델에서 bmi 변수가 한 단위 증가할 때와
## 세 단위 증가할 때, 질병 발생의 오즈비 값은 각각 얼마인가
# 아까 구한 bmi의 계수 값을 이용 ** model.params['bmi']를 이용하여 직접 값 추출 가능 **
# 로지스틱 회귀 모델의 계수 = 로그 오즈의 변화향 / 오즈 -> 오즈비 변환을 위해 np.exp() 함수로 계수 값을 지수화하여 계산 수행

## 1. numpy 라이브러리
import numpy as np # 나 이거 써봤는데 - 익숙함
print(model.params['bmi']) ## 'bmi'의 계수값 직접 추출 (결과 : 0.056322754028830115)
print(np.exp(model.params['bmi'])) ## numpy 라이브러리 np.exp() 함수 인자로 독립 변수의 계수값을 넣어줌 -> 지수 변환 = 오즈비

# 결과 : 1.057939082740907
# 결과값이 의미하는 것 : bmi가 한 단위 증가하면 질병 발생(종속 변수)의 오즈비는 약 1.0579

## 2. 문제에서 요구 - bmi 3 단위 증가 시 오즈비
## 증가하는 단위만큼 계수에 배수 적용하여 지수 변환
print(np.exp(model.params['bmi']*3)) # 결과 : 1.184082558017788

0.056322754028830115
1.057939082740907
1.184082558017788


In [20]:
#### 범주형 변수의 오즈비 - 인코딩 해야하나
### 자동으로 원-핫 인코딩 수행
### 3 가지 값에 적용한다고 가정, 첫 번째 값을 기준으로 다른 두 값을 비교한 결과값을 취급 (3가지 -> 결과는 2가지)

## 문제 조건 : 문제 1에서 적합한 로지스틱 회귀 모델에서 lifestyle 변수가 'A'인 사람 대비
## 'S'인 사람의 오즈비를 구하시오 (반올림하여 소수 둘 째 자리까지 작성)
print(model.summary()) ## 통계 요약 확인 좀 하고.

# 알아야 할 거 : lifestyle 변수 부분 ([T.M], [T.S] -> 변수가 3가지로 구성되어 결과 변수로 2가지가 생성된 것)
# 문제에서 'S'인 사람 오즈비 요구 : 'S'에 해당하는 계수 추출 / 지수화 -> 오즈비 계산

print(np.exp(model.params['lifestyle[T.S]']))
# 결과 : 1.0829249581859632
# 의미 : 기준 'A' 대비 'S' 오즈비 약 1.082 - 'A' 사람 대비 'S' 사람의 질병 발생 오즈가 약 1.8배 높음

                           Logit Regression Results                           
Dep. Variable:                disease   No. Observations:                 1000
Model:                          Logit   Df Residuals:                      994
Method:                           MLE   Df Model:                            5
Date:                Tue, 16 Jun 2026   Pseudo R-squ.:                 0.05341
Time:                        18:05:52   Log-Likelihood:                -641.39
converged:                       True   LL-Null:                       -677.58
Covariance Type:            nonrobust   LLR p-value:                 3.284e-14
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -1.9881      0.306     -6.492      0.000      -2.588      -1.388
lifestyle[T.M]     0.0250      0.167      0.150      0.881      -0.302       0.352
lifestyle[T.S]     0.0797      0.168

In [31]:
## 로지스틱 회귀 모델의 예측
## 기존 데이터로 로지 회귀 모델을 학습하고, 새 데이터에 예측 수행할 수 있음
# 함수 statsmodels.predict() 사용

## 문제 조건 : 주어진 데이터를 train(처음 700행) test(나머지 300행)로 분할하시오. age, bmi, smoker lifestyle을 독립 변수로,
## disease를 종속 변수로 하는 로지스틱 회귀 모델을 train 데이터로 적합하시오. 적합된 모델로 test 데이터 에측 시 질병 발생 확률이 가장 높은 사람의 bmi?
## (반올림하여 소수 첫째 자리까지)

# 1. 라이브러리 및 데이터 분할
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part3/ch5/health_survey2.csv')
train = df.iloc[:700]  ## 처음부터 700까지 (0부터 시작이므로 699까지 잘릴거임) : 슬라이싱
test = df.iloc[700:]  ## 700부터 끝까지
print(train.shape, test.shape)  # 결과 : (700, 6) (300, 6) 행 확인

# 2. 로지스틱 회귀 사용 (logit임 ols 아님)
from statsmodels.formula.api import logit
model = logit('disease~age+bmi+smoker+lifestyle', data=train).fit()  ## 적합(학습) 수행 : 옵션값 data= 에 문제에서 요구한 train 넣어야 함

# 3. test 데이터 예측
pred = model.predict(test) ## 얘도 predict() 인자값으로 문제에서 요구한 test 사용
print(pred.sort_values()) # 왜 기억이 안 나나 했더니 모르는거네 : sort_values() 사용해서 질병 발생 확률을 오름차순으로 출력 - 가장 마지막(확률 높은) 데이터 행 840 확인
print(test.loc[840, 'bmi']) ## test 데이터에서 가장 확률이 높은 사람[840]행의 'bmi' 값 추출

# 결과 : 42.2

## 조금 더 쉬운 방법 idxmax() 함수 사용
from statsmodels.formula.api import logit
model = logit('disease~age+bmi+smoker+lifestyle', data=train).fit()
pred = model.predict(test)
max_idx = pred.idxmax() ## 가장 큰 값을 갖는 인덱스 추출
test.loc[max_idx]

(700, 6) (300, 6)
Optimization terminated successfully.
         Current function value: 0.637966
         Iterations 5
773    0.240990
996    0.252992
863    0.287612
777    0.313125
861    0.323196
         ...   
852    0.874378
889    0.877035
946    0.878470
985    0.896081
840    0.900602
Length: 300, dtype: float64
42.2
Optimization terminated successfully.
         Current function value: 0.637966
         Iterations 5


,840
age,72
bmi,42.2
smoker,1
activity_level,0
disease,0
lifestyle,S


In [39]:
## 30분에 20분 추가요. 곧 끝남
## 문제 조건 : 문제 4에서 train 데이터로 적합시킨 모델의 결과를 확인하여, p-value가 0.05 이하인 독립 변수만 선택하시오
## 선택된 변수들만 사용하여 로지스틱 회귀 모델을 재적합 수행, 이 모델로 test 데이터 예측한 후 확률이 0.4 초과인 사람을
## 질병 있음(disease)으로 분류했을 때 민감도?
# 민감도 : 실제 양성(disease=1) 사람 중 모델이 양성으로 올바르게 예측한 비율 (=재현율)

## p-value 조건이 주어졌으므로 방금 문제에서 만든 모델의 summary() 확인
print(model.summary()) ## p-value < 0.05인 데이터 = Intercept, age, bmi (인데 Intercept는 제외해야지)

# 1. 로지스틱 회귀 모델 재적합 수행
from statsmodels.formula.api import logit
model = logit('disease~age+bmi', data=train).fit()
pred = model.predict(test)

# 2. 조건 필터링 : 문제에서 확률이 0.4 초과인 사람 (기준값 : 0.4)
pred = (pred > 0.4).astype(int) # 조건 적용 (조건).astype(자료형)

# 3. 민감도 계산
## 민감도 라이브러리 sklearn.metrics import recall_score   ** 민감도 : recall_score
from sklearn.metrics import recall_score
print(recall_score(test['disease'], pred)) # recall_score(분류할 결과, 대상) : 인자에 test 데이터 프레임 사용 / 아까 필터링한 조건의 pred
## print(recall_score(test['disease'], pred))
## 결과 : 0.96

                           Logit Regression Results                           
Dep. Variable:                disease   No. Observations:                  700
Model:                          Logit   Df Residuals:                      697
Method:                           MLE   Df Model:                            2
Date:                Tue, 16 Jun 2026   Pseudo R-squ.:                 0.05058
Time:                        18:31:54   Log-Likelihood:                -449.84
converged:                       True   LL-Null:                       -473.80
Covariance Type:            nonrobust   LLR p-value:                 3.909e-11
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.8710      0.343     -5.455      0.000      -2.543      -1.199
age            0.0197      0.004      4.395      0.000       0.011       0.028
bmi            0.0535      0.010      5.129      0.0

In [ ]:
import pandas as pd
# df = pd.read_csv("health_survey2.csv")
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part3/ch5/health_survey2.csv")

print(df.head())

   age   bmi  smoker  activity_level  disease lifestyle
0   62  35.2       0               0        1         S
1   65  18.6       0               2        1         A
2   71  33.2       0               1        1         M
3   18  37.1       1               2        0         A
4   21  17.6       0               0        0         S


In [ ]:
from statsmodels.formula.api import logit

model = logit('disease ~ age + bmi + smoker + lifestyle', data=df).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.641392
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                disease   No. Observations:                 1000
Model:                          Logit   Df Residuals:                      994
Method:                           MLE   Df Model:                            5
Date:                Tue, 26 May 2026   Pseudo R-squ.:                 0.05341
Time:                        10:44:08   Log-Likelihood:                -641.39
converged:                       True   LL-Null:                       -677.58
Covariance Type:            nonrobust   LLR p-value:                 3.284e-14
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -1.9881      0.306     -6.492      0.000      -2.588      -1.388
lifestyle[T.M]   

In [ ]:
import numpy as np
print(model.params['bmi'])
print(np.exp(model.params['bmi']))

0.0563227540288301
1.057939082740907


In [ ]:
print(np.exp(model.params['bmi']*3))

1.184082558017788


In [ ]:
print(np.exp(model.params['lifestyle[T.S]']))

1.0829249581859632


In [ ]:
# [TIP]'S'를 기준 범주로 설정
from statsmodels.formula.api import logit
model = logit('disease ~ age + bmi + smoker + C(lifestyle, Treatment("S"))', data=df).fit()
print(model.summary())


Optimization terminated successfully.
         Current function value: 0.641392
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                disease   No. Observations:                 1000
Model:                          Logit   Df Residuals:                      994
Method:                           MLE   Df Model:                            5
Date:                Tue, 26 May 2026   Pseudo R-squ.:                 0.05341
Time:                        10:44:08   Log-Likelihood:                -641.39
converged:                       True   LL-Null:                       -677.58
Covariance Type:            nonrobust   LLR p-value:                 3.284e-14
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -1.9084      0.316     -6

In [ ]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part3/ch5/health_survey2.csv")

train = df.iloc[:700]
test = df.iloc[700:]
print(train.shape, test.shape)

(700, 6) (300, 6)


In [ ]:
# 방법1
from statsmodels.formula.api import logit

model = logit('disease ~ age + bmi + smoker + lifestyle', data=train).fit()
pred = model.predict(test)
print(pred.sort_values()) #인덱스 840 확인
print(test.loc[840, 'bmi'])

# 정답 42.2

Optimization terminated successfully.
         Current function value: 0.637966
         Iterations 5
773    0.240990
996    0.252992
863    0.287612
777    0.313125
861    0.323196
         ...   
852    0.874378
889    0.877035
946    0.878470
985    0.896081
840    0.900602
Length: 300, dtype: float64
42.2


In [ ]:
# 방법2
from statsmodels.formula.api import logit

model = logit('disease ~ age + bmi + smoker + lifestyle', data=train).fit()
pred = model.predict(test)
max_idx = pred.idxmax()
print(test.loc[max_idx])

# 정답 42.2

Optimization terminated successfully.
         Current function value: 0.637966
         Iterations 5
age                 72
bmi               42.2
smoker               1
activity_level       0
disease              0
lifestyle            S
Name: 840, dtype: object


In [ ]:
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:                disease   No. Observations:                  700
Model:                          Logit   Df Residuals:                      694
Method:                           MLE   Df Model:                            5
Date:                Tue, 26 May 2026   Pseudo R-squ.:                 0.05746
Time:                        10:44:08   Log-Likelihood:                -446.58
converged:                       True   LL-Null:                       -473.80
Covariance Type:            nonrobust   LLR p-value:                 1.694e-10
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -2.0554      0.368     -5.579      0.000      -2.778      -1.333
lifestyle[T.M]     0.0368      0.198      0.186      0.852      -0.351       0.425
lifestyle[T.S]     0.3639      0.201

In [ ]:
from statsmodels.formula.api import logit

model = logit('disease ~ age + bmi', data=train).fit()
pred = model.predict(test)
pred = (pred > 0.4).astype(int)

from sklearn.metrics import recall_score
print(recall_score(test['disease'], pred))

Optimization terminated successfully.
         Current function value: 0.642623
         Iterations 5
0.96
